# RetailPulse -- Optuna Hyperparameter Tuning

**Objective:** Use Optuna to optimize XGBoost churn model hyperparameters.

In [1]:
import os, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use("seaborn-v0_8-whitegrid")
FIGURES_DIR = os.path.join("..", "reports", "figures")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(FIGURES_DIR, exist_ok=True)
def save_fig(fig, name):
    fig.savefig(os.path.join(FIGURES_DIR, name), dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig); print(f"Saved: {name}")


In [2]:
rfm = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_rfm.csv"))
rfm["is_churned"] = (rfm["recency"] > 90).astype(int)
feature_cols = ["recency", "frequency", "monetary", "r_score", "f_score", "m_score", "rfm_score"]
X = rfm[feature_cols].values
y = rfm["is_churned"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")


Train: 3449 | Test: 863


## Baseline Model

In [3]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)
baseline_params = {"max_depth": 5, "learning_rate": 0.1, "objective": "binary:logistic", "eval_metric": "logloss", "seed": 42}
baseline_model = xgb.train(baseline_params, dtrain, num_boost_round=200, verbose_eval=False)
baseline_auc = roc_auc_score(y_test, baseline_model.predict(dtest))
print(f"Baseline ROC AUC: {baseline_auc:.4f}")


Baseline ROC AUC: 1.0000


## Optuna Optimization

In [4]:
def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "objective": "binary:logistic", "eval_metric": "logloss", "seed": 42,
    }
    n_rounds = trial.suggest_int("n_estimators", 50, 300)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []
    for tr_idx, val_idx in cv.split(X_train, y_train):
        dt = xgb.DMatrix(X_train[tr_idx], label=y_train[tr_idx])
        dv = xgb.DMatrix(X_train[val_idx], label=y_train[val_idx])
        m = xgb.train(params, dt, num_boost_round=n_rounds, verbose_eval=False)
        aucs.append(roc_auc_score(y_train[val_idx], m.predict(dv)))
    return np.mean(aucs)

study = optuna.create_study(direction="maximize", study_name="xgb_churn")
study.optimize(objective, n_trials=50, show_progress_bar=False)
print(f"Best trial AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")


Best trial AUC: 1.0000
Best params: {'max_depth': 6, 'learning_rate': 0.055208070989012946, 'min_child_weight': 7, 'subsample': 0.8789635011353805, 'colsample_bytree': 0.902755047752842, 'gamma': 4.731773113461227, 'reg_alpha': 7.074773646791061, 'reg_lambda': 7.989557827039184, 'n_estimators': 153}


In [5]:
# Train final model with best params
best = study.best_params.copy()
n_rounds = best.pop("n_estimators")
best["objective"] = "binary:logistic"
best["eval_metric"] = "logloss"
best["seed"] = 42
tuned_model = xgb.train(best, dtrain, num_boost_round=n_rounds, verbose_eval=False)
tuned_auc = roc_auc_score(y_test, tuned_model.predict(dtest))
improvement = (tuned_auc - baseline_auc) / baseline_auc * 100
print(f"Baseline AUC: {baseline_auc:.4f}")
print(f"Tuned AUC:    {tuned_auc:.4f}")
print(f"Improvement:  {improvement:+.2f}%")


Baseline AUC: 1.0000
Tuned AUC:    1.0000
Improvement:  -0.00%


In [6]:
# Optimization history
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
trials_df = study.trials_dataframe()
axes[0].plot(trials_df["number"], trials_df["value"], "o-", color="#3498db", markersize=4, linewidth=1)
axes[0].axhline(baseline_auc, color="#e74c3c", linestyle="--", label=f"Baseline ({baseline_auc:.4f})")
axes[0].axhline(study.best_value, color="#27ae60", linestyle="--", label=f"Best ({study.best_value:.4f})")
axes[0].set_xlabel("Trial"); axes[0].set_ylabel("ROC AUC"); axes[0].set_title("Optimization History"); axes[0].legend()

param_imp = optuna.importance.get_param_importances(study)
params_sorted = sorted(param_imp.items(), key=lambda x: x[1], reverse=True)
axes[1].barh([p[0] for p in params_sorted], [p[1] for p in params_sorted], color="#9b59b6")
axes[1].set_xlabel("Importance"); axes[1].set_title("Hyperparameter Importance")
axes[1].invert_yaxis()
fig.suptitle("Optuna XGBoost Tuning (50 Trials)", fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout(); save_fig(fig, "37_optuna_optimization.png"); plt.show()


Saved: 37_optuna_optimization.png


In [7]:
# Save results
results = pd.DataFrame({"param": list(study.best_params.keys()), "value": [str(v) for v in study.best_params.values()]})
results.to_csv(os.path.join(PROCESSED_DIR, "optuna_best_params.csv"), index=False)
print("Saved: optuna_best_params.csv")
print("\nOPTUNA TUNING COMPLETE")


Saved: optuna_best_params.csv

OPTUNA TUNING COMPLETE
